# NLP Feature Extraction for Spotify Songs

This notebook extracts NLP features from song lyrics and metadata using pre-trained sentence embeddings. These features will be used in an ensemble model for song recommendation.

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import os

# Paths
DATA_DIR = "/home/build/martin/spotify_x5000"
METADATA_PATH = os.path.join(DATA_DIR, "spotify_5000.parquet")
OUTPUT_PATH = os.path.join(DATA_DIR, "nlp_features.csv")

In [ ]:
print("Loading metadata...")
df = pd.read_parquet(METADATA_PATH)
print(f"Loaded {len(df)} tracks.")

In [ ]:
print("Preprocessing text for embeddings...")
# Combine track name, artist name, and lyrics for a better semantic representation
# This helps if lyrics are missing or for better context
df['combined_text'] = (
    df['track_name'].fillna('') + " by " + 
    df['artist_name'].fillna('') + ". Lyrics: " + 
    df['lyrics'].fillna("")
)

display(df[['track_id', 'combined_text']].head())

In [ ]:
print("Initializing SentenceTransformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings (this may take a few minutes)...")
embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)

print(f"Generated embeddings with shape: {embeddings.shape}")

In [ ]:
print("Preparing features dataframe...")
nlp_features_df = pd.DataFrame(
    embeddings, 
    columns=[f'nlp_feat_{i}' for i in range(embeddings.shape[1])]
)
nlp_features_df['track_id'] = df['track_id'].values

print("Saving nlp_features.csv...")
nlp_features_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")